In [39]:
import numpy as np
import pandas as pd

In [40]:
df = pd.DataFrame(
    [[8, 8, 4], [7, 9, 5], [6, 10, 6], [5, 12, 7]],
    columns=["cgpa", "profile_score", "lpa"],
)
df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


In [41]:
def init_parameters(layer_dims: list):
    np.random.seed(3)

    parameters = {}
    L = len(layer_dims)

    for l in range(1, L):
        parameters["W" + str(l)] = np.ones((layer_dims[l - 1], layer_dims[l])) * 0.1
        parameters["b" + str(l)] = np.zeros((layer_dims[l], 1))

    return parameters

In [42]:
init_parameters([2, 2, 1])["W2"]

array([[0.1],
       [0.1]])

In [43]:
def linear_forward(A_prev, W, b):

    Z = np.dot(W.T, A_prev) + b
    return Z

In [44]:
def L_layer_forward(X, parameters):  # forward propagation
    A = X
    L = len(parameters) // 2

    for l in range(1, L + 1):
        A_prev = A
        Wl = parameters["W" + str(l)]
        bl = parameters["b" + str(l)]

        # print(f"A{str(l - 1)}: {A_prev}")
        # print(f"W{str(l)}: {Wl}")
        # print(f"b{str(l)}: {bl}")
        # print("--" * 20)

        A = linear_forward(A_prev, Wl, bl)
        # print(f"A{str(l)}: {A}")
        # print("**" * 20)

    return A, A_prev

In [45]:
X = df[["cgpa", "profile_score"]].values[0].reshape(2, 1)
y = df[["lpa"]].values[0][0]

# Paramter initialization
params = init_parameters([2, 2, 1])
y_hat, A1 = L_layer_forward(X, params)

In [46]:
y_hat = y_hat[0][0]
y_hat

0.32000000000000006

#### Update weights
- calculate all 9 derivatives

In [47]:
def update_params(parameters, y, y_hat, A1, X):
    lr = 0.001

    parameters["W2"][0][0] = parameters["W2"][0][0] + (lr * 2 * (y - y_hat) * A1[0][0])
    parameters["W2"][1][0] = parameters["W2"][1][0] + (lr * 2 * (y - y_hat) * A1[1][0])
    parameters["b2"][0][0] = parameters["W2"][1][0] + (lr * 2 * (y - y_hat))

    parameters["W1"][0][0] = parameters["W1"][0][0] + (
        lr * 2 * (y - y_hat) * parameters["W2"][0][0] * X[0][0]
    )
    parameters["W1"][0][1] = parameters["W1"][0][1] + (
        lr * 2 * (y - y_hat) * parameters["W2"][0][0] * X[1][0]
    )
    parameters["b1"][0][0] = parameters["b1"][0][0] + (
        lr * 2 * (y - y_hat) * parameters["W2"][0][0]
    )

    parameters["W1"][1][0] = parameters["W1"][1][0] + (
        lr * 2 * (y - y_hat) * parameters["W2"][1][0] * X[0][0]
    )
    parameters["W1"][1][1] = parameters["W1"][1][1] + (
        lr * 2 * (y - y_hat) * parameters["W2"][1][0] * X[1][0]
    )
    parameters["b1"][1][0] = parameters["b1"][1][0] + (
        lr * 2 * (y - y_hat) * parameters["W2"][1][0]
    )

In [48]:
update_params(params, y, y_hat, A1, X)

In [49]:
params

{'W1': array([[0.10658137, 0.10658137],
        [0.10658137, 0.10658137]]),
 'b1': array([[0.00082267],
        [0.00082267]]),
 'W2': array([[0.111776],
        [0.111776]]),
 'b2': array([[0.119136]])}

`One interation completed`

---

### Epochs implementation

In [ ]:
params = init_parameters([2, 2, 1])
epochs = 5

for i in range(epochs):
    loss = []

    for j in range(df.shape[0]):
        X = df[["cgpa", "profile_score"]].values[j].reshape(2, 1)
        y = df[["lpa"]].values[j][0]

        # Parameters initializaition

        y_hat, A1 = L_layer_forward(X, params)
        y_hat = y_hat[0][0]

        update_params(params, y, y_hat, A1, X)
        loss.append((y - y_hat) ** 2)

    print(f"Epoch: {i + 1} | Loss: {np.array(loss).mean()}")

params

Epoch: 1 | Loss: 25.321744156025517
Epoch: 2 | Loss: 18.320004165722047
Epoch: 3 | Loss: 9.473661050729628
Epoch: 4 | Loss: 3.2520938634031613
Epoch: 5 | Loss: 1.3407132589299962


{'W1': array([[0.26507636, 0.38558861],
        [0.27800387, 0.40980287]]),
 'b1': array([[0.02749056],
        [0.02974394]]),
 'W2': array([[0.41165744],
        [0.48302736]]),
 'b2': array([[0.48646246]])}

---
# Keras Implementation

In [53]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [54]:
model = Sequential()

model.add(Dense(2, activation="linear", input_dim=2))
model.add(Dense(1, activation="linear"))

/home/slayer0x10/dev/ml/ml-study/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1771420476.252013  582291 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3380 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050, pci bus id: 0000:10:00.0, compute capability: 8.6


In [55]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 2)              │             6 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

In [56]:
### Change weights to 0.1 and baises to 0.

new_weights = [
    np.array([[0.1, 0.1], [0.1, 0.1]], dtype=np.float32),
    np.array([0.0, 0.0], dtype=np.float32),
    np.array([[0.1], [0.1]], dtype=np.float32),
    np.array([0.0], dtype=np.float32),
]

model.set_weights(new_weights)

In [57]:
model.get_weights()

[array([[0.1, 0.1],
        [0.1, 0.1]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[0.1],
        [0.1]], dtype=float32),
 array([0.], dtype=float32)]

In [58]:
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss="mean_squared_error", optimizer=optimizer)

In [59]:
model.fit(df.iloc[:, 0:-1].values, df["lpa"].values, epochs=75, verbose=1, batch_size=1)

Epoch 1/75


2026-02-18 18:49:56.424396: I external/local_xla/xla/service/service.cc:163] XLA service 0x7fb76c008280 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-18 18:49:56.424414: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050, Compute Capability 8.6
2026-02-18 18:49:56.446262: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-18 18:49:56.509894: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 90300
I0000 00:00:1771420796.622530  610435 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 27.8873  
Epoch 2/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 27.5658 
Epoch 3/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 27.2614 
Epoch 4/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 26.8970 
Epoch 5/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 26.5422 
Epoch 6/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 26.1824 
Epoch 7/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 25.8228 
Epoch 8/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 25.4714 
Epoch 9/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 25.0492 
Epoch 10/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 24.6267 
Epoch 11/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 24.2428 
Epoch 12/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 23.8183 
Epoch 13/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 23.3676 
Epoch 14/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 22.9260 
Epoch 15/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 22.4633 
Epoch 16/75
4/4 ━━━━━━━━━━━━━

In [60]:
model.get_weights()

[array([[0.37366578, 0.37366578],
        [0.36565182, 0.36565182]], dtype=float32),
 array([0.27235466, 0.27235466], dtype=float32),
 array([[0.37296113],
        [0.37296113]], dtype=float32),
 array([0.20485015], dtype=float32)]